In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("../data/raw")
files = {
    "caract": "caract-2023.csv",
    "lieux": "lieux-2023.csv",
    "usagers": "usagers-2023.csv",
    "vehicules": "vehicules-2023.csv",
}

def load_raw(path):
    """Load a BAAC CSV as text and normalise whitespace and missing markers."""
    df = pd.read_csv(path, sep=";", dtype=str, encoding="utf-8-sig")
    df.columns = df.columns.str.strip()
    # Strip regular, non-breaking and narrow non-breaking spaces at both ends of each cell
    df = df.apply(lambda s: s.str.strip().str.strip("\u00a0\u202f"))
    return df.replace({"": np.nan, "N/A": np.nan})

dfs = {k: load_raw(RAW / f) for k, f in files.items()}

# Harmonise the accident key name across years (some files use "Accident_Id")
for df in dfs.values():
    df.rename(columns={"Accident_Id": "Num_Acc"}, inplace=True)

# Clean IDs: remove all inner spaces (regular, non-breaking, narrow)
for df in dfs.values():
    for col in ["id_usager", "id_vehicule"]:
        if col in df.columns:
            df[col] = df[col].str.replace("[\\s\u00a0\u202f]", "", regex=True)

# Coordinates: comma decimal -> float
for col in ["lat", "long"]:
    dfs["caract"][col] = pd.to_numeric(
        dfs["caract"][col].str.replace(",", ".", regex=False), errors="coerce"
    )

for k, df in dfs.items():
    print(f"--- {k}: {df.shape}")
    print(list(df.columns))

dfs["caract"][["lat", "long"]].describe()

--- caract: (54822, 15)
['Num_Acc', 'jour', 'mois', 'an', 'hrmn', 'lum', 'dep', 'com', 'agg', 'int', 'atm', 'col', 'adr', 'lat', 'long']
--- lieux: (70860, 18)
['Num_Acc', 'catr', 'voie', 'v1', 'v2', 'circ', 'nbv', 'vosp', 'prof', 'pr', 'pr1', 'plan', 'lartpc', 'larrout', 'surf', 'infra', 'situ', 'vma']
--- usagers: (125789, 16)
['Num_Acc', 'id_usager', 'id_vehicule', 'num_veh', 'place', 'catu', 'grav', 'sexe', 'an_nais', 'trajet', 'secu1', 'secu2', 'secu3', 'locp', 'actp', 'etatp']
--- vehicules: (93585, 11)
['Num_Acc', 'id_vehicule', 'num_veh', 'senc', 'catv', 'obs', 'obsm', 'choc', 'manv', 'motor', 'occutc']


,lat,long
count,54822.000000,54822.000000
mean,44.250545,1.830744
std,12.419628,18.959204
min,-23.372504,-176.207000
25%,44.790693,1.172116
50%,47.859229,2.384819
75%,48.861237,4.734031
max,51.047490,168.095670


In [3]:
# --- Build one table: 1 row = 1 person involved (metropolitan France, 2023) ---
lieux_main = (dfs["lieux"].assign(catr_num=pd.to_numeric(dfs["lieux"]["catr"], errors="coerce"))
              .sort_values(["Num_Acc", "catr_num"])
              .drop_duplicates(subset="Num_Acc"))

df = (dfs["usagers"]
      .merge(dfs["caract"], on="Num_Acc", how="left")
      .merge(lieux_main[["Num_Acc", "catr", "vma"]], on="Num_Acc", how="left")
      .merge(dfs["vehicules"][["Num_Acc", "id_vehicule", "catv"]], on=["Num_Acc", "id_vehicule"], how="left"))
assert len(df) == len(dfs["usagers"]), "Join created duplicates!"

df = df[(df["dep"].str.len() <= 2) & (df["grav"] != "-1")].copy()   # metropolitan France, known severity

# --- Features ---
df["killed"] = df["grav"] == "2"
df["hour"] = df["hrmn"].str[:2].astype(int)
df["night"] = df["lum"].isin(["3", "4", "5"])
date = pd.to_datetime(dict(year=df["an"].astype(int), month=df["mois"].astype(int), day=df["jour"].astype(int)))
df["weekend"] = date.dt.dayofweek >= 5
df["age"] = 2023 - pd.to_numeric(df["an_nais"], errors="coerce")
df["age_group"] = pd.cut(df["age"], [-1, 17, 24, 34, 44, 54, 64, 74, 120],
                         labels=["0-17", "18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75+"])
df["area"] = df["agg"].map({"1": "Outside urban area", "2": "Urban area"})
df["road"] = df["catr"].map({"1": "Motorway", "2": "National road", "3": "Departmental road",
                             "4": "Municipal road", "7": "Metropolitan road"}).fillna("Other")

catv = pd.to_numeric(df["catv"], errors="coerce")
df["mode"] = np.select(
    [df["catu"] == "3", catv.isin([1, 80]), catv.isin([50, 60]),
     catv.isin([2, 30, 31, 32, 33, 34, 41, 42, 43]), catv.isin([7, 10]),
     catv.isin([13, 14, 15, 16, 17, 37, 38])],
    ["Pedestrian", "Bicycle", "E-scooter / PMD", "Powered two-wheeler", "Car / van", "Heavy vehicle / bus"],
    default="Other")

print(len(df), "people |", df["killed"].sum(), "killed")  # should be 3,167 killed

118101 people | 3167 killed


In [4]:
# --- Scan: share of deaths vs. lethality for each candidate angle ---
avg = df["killed"].mean() * 100

def scan(col):
    t = df.groupby(col, observed=True).agg(people=("killed", "size"), killed=("killed", "sum"))
    t["killed_per_100"] = (t["killed"] / t["people"] * 100).round(2)
    t["vs_avg"] = (t["killed_per_100"] / avg).round(1)          # 2.0 = twice as lethal as average
    t["share_of_deaths_%"] = (t["killed"] / t["killed"].sum() * 100).round(1)
    return t.sort_values("killed", ascending=False)

print(f"Average: {avg:.2f} killed per 100 people involved\n")
for col in ["area", "road", "night", "weekend", "mode", "age_group"]:
    print(f"===== {col}")
    display(scan(col))

Average: 2.68 killed per 100 people involved

===== area


,people,killed,killed_per_100,vs_avg,share_of_deaths_%
area,,,,,
Outside urban area,46346,2139,4.62,1.7,67.5
Urban area,71755,1028,1.43,0.5,32.5


===== road


,people,killed,killed_per_100,vs_avg,share_of_deaths_%
road,,,,,
Departmental road,45204,1935,4.28,1.6,61.1
Municipal road,47983,648,1.35,0.5,20.5
Motorway,13775,269,1.95,0.7,8.5
National road,6414,192,2.99,1.1,6.1
Metropolitan road,3398,79,2.32,0.9,2.5
Other,1327,44,3.32,1.2,1.4


===== night


...

,people,killed,killed_per_100,vs_avg,share_of_deaths_%
weekend,,,,,
False,85715,2118,2.47,0.9,66.9
True,32386,1049,3.24,1.2,33.1


===== mode


,people,killed,killed_per_100,vs_avg,share_of_deaths_%
mode,,,,,
Car / van,75630,1637,2.16,0.8,51.7
Powered two-wheeler,18631,706,3.79,1.4,22.3
Pedestrian,8992,435,4.84,1.8,13.7
Bicycle,5755,221,3.84,1.4,7.0
Other,2064,70,3.39,1.3,2.2
Heavy vehicle / bus,4324,50,1.16,0.4,1.6
E-scooter / PMD,2705,48,1.77,0.7,1.5


===== age_group


,people,killed,killed_per_100,vs_avg,share_of_deaths_%
age_group,,,,,
75+,6143,536,8.73,3.3,16.9
18-24,22010,495,2.25,0.8,15.6
25-34,23104,469,2.03,0.8,14.8
55-64,12172,397,3.26,1.2,12.5
45-54,15987,392,2.45,0.9,12.4
35-44,18432,364,1.97,0.7,11.5
65-74,7182,364,5.07,1.9,11.5
0-17,10721,150,1.40,0.5,4.7


In [5]:
# --- Cross-scan: deaths and lethality for each combination of two angles ---
def cross(row, col):
    t = df.groupby([row, col], observed=True).agg(people=("killed", "size"), killed=("killed", "sum"))
    t["killed_per_100"] = (t["killed"] / t["people"] * 100).round(2)
    # Wide tables: one row per `row` category, one column per `col` category
    killed = t["killed"].unstack(fill_value=0)
    killed["Total"] = killed.sum(axis=1)                         # row total, to read each column as a share of it
    killed = killed.sort_values("Total", ascending=False)
    rate = t["killed_per_100"].unstack().reindex(killed.index)  # same row order; NaN = no one involved
    return killed, rate

for row, col in [("age_group", "mode"), ("road", "mode")]:
    killed, rate = cross(row, col)
    print(f"===== {row} x {col}: killed")
    display(killed)
    print(f"===== {row} x {col}: killed per 100 people involved")
    display(rate)

===== age_group x mode: killed


mode,Bicycle,Car / van,E-scooter / PMD,Heavy vehicle / bus,Other,Pedestrian,Powered two-wheeler,Total
age_group,,,,,,,,
75+,58,289,1,1,8,169,10,536
18-24,8,326,12,1,10,28,110,495
25-34,14,236,7,5,11,39,157,469
55-64,40,178,3,14,8,37,117,397
45-54,20,195,5,17,5,54,96,392
35-44,13,163,10,8,6,35,129,364
65-74,57,188,3,4,17,57,38,364
0-17,11,62,7,0,5,16,49,150


===== age_group x mode: killed per 100 people involved


mode,Bicycle,Car / van,E-scooter / PMD,Heavy vehicle / bus,Other,Pedestrian,Powered two-wheeler
age_group,,,,,,,
75+,17.96,6.85,10.00,2.70,16.33,12.12,8.85
18-24,1.07,2.15,1.60,0.37,3.34,3.19,2.82
25-34,1.25,1.57,1.18,0.68,3.56,4.13,3.62
55-64,5.04,2.37,2.91,2.00,3.28,3.78,6.33
45-54,2.34,1.92,2.16,1.71,1.89,5.59,3.78
35-44,1.64,1.33,2.79,0.83,1.83,4.04,4.53
65-74,11.05,3.91,12.00,3.54,14.05,5.72,6.26
0-17,2.03,1.31,1.22,0.00,2.27,0.82,2.17


===== road x mode: killed


mode,Bicycle,Car / van,E-scooter / PMD,Heavy vehicle / bus,Other,Pedestrian,Powered two-wheeler,Total
road,,,,,,,,
Departmental road,137,1081,20,17,38,167,475,1935
Municipal road,71,206,25,6,24,173,143,648
Motorway,2,163,0,18,3,39,44,269
National road,2,137,1,9,1,23,19,192
Metropolitan road,6,34,2,0,1,13,23,79
Other,3,16,0,0,3,20,2,44


===== road x mode: killed per 100 people involved


mode,Bicycle,Car / van,E-scooter / PMD,Heavy vehicle / bus,Other,Pedestrian,Powered two-wheeler
road,,,,,,,
Departmental road,7.79,3.58,3.34,1.03,4.21,7.05,6.15
Municipal road,1.87,0.78,1.26,0.52,2.70,2.98,1.77
Motorway,33.33,1.48,0.00,1.82,3.00,37.86,2.86
National road,6.06,2.70,7.14,2.24,1.72,17.16,2.72
Metropolitan road,4.80,1.45,2.47,0.00,1.54,5.83,4.94
Other,6.67,2.37,0.00,0.00,6.25,5.48,1.48


### Key insights (2023, metropolitan France)

1. **Where — departmental roads**: 61% of deaths (lethality ×1.6). They also concentrate
   67% of powered two-wheeler deaths and 62% of cyclist deaths → priority for **départements**.
2. **Urban pedestrians**: pedestrians are the first victims on municipal roads
   (173 deaths, 40% of all pedestrian deaths) → priority for **communes**.
3. **Seniors**: people aged 65+ are 11% of those involved but **52% of pedestrians killed and
   52% of cyclists killed**. Pedestrians aged 75+ have a lethality of 12.1 per 100 (×4.5 the average),
   partly reflecting physical frailty.
4. **Young adults**: 18–24 year-olds are the age group with the most car-occupant deaths (326);
   56% of powered two-wheeler deaths are aged 18–44.
5. Night and weekend effects are moderate (×1.2) → used as filters, not as a main section.



In [6]:
# --- Multi-year check: reload 2019-2024 with the same loading and join logic ---
YEARS = range(2019, 2025)
TABLES = ["caract", "lieux", "usagers", "vehicules"]
CODE_COLS = {"caract": ["lum", "agg", "int"], "lieux": ["catr"],
             "usagers": ["catu", "grav", "sexe"], "vehicules": ["catv"]}   # codes used by the analysis

def build_year(year):
    """Same steps as above, for one year: 1 row = 1 person involved, metropolitan France, known severity."""
    d = {t: load_raw(RAW / f"{t}-{year}.csv") for t in TABLES}
    for t in d.values():
        t.rename(columns={"Accident_Id": "Num_Acc"}, inplace=True)
        for col in ["id_usager", "id_vehicule"]:
            if col in t.columns:
                t[col] = t[col].str.replace("[\\s\u00a0\u202f]", "", regex=True)

    lieux_main = (d["lieux"].assign(catr_num=pd.to_numeric(d["lieux"]["catr"], errors="coerce"))
                  .sort_values(["Num_Acc", "catr_num"])
                  .drop_duplicates(subset="Num_Acc"))
    p = (d["usagers"]
         .merge(d["caract"], on="Num_Acc", how="left")
         .merge(lieux_main[["Num_Acc", "catr"]], on="Num_Acc", how="left")
         .merge(d["vehicules"][["Num_Acc", "id_vehicule", "catv"]], on=["Num_Acc", "id_vehicule"], how="left"))
    assert len(p) == len(d["usagers"]), f"{year}: join created duplicates!"
    p = p[(p["dep"].str.len() <= 2) & (p["grav"] != "-1")].copy()

    p["killed"] = p["grav"] == "2"
    p["age"] = year - pd.to_numeric(p["an_nais"], errors="coerce")
    catv = pd.to_numeric(p["catv"], errors="coerce")
    p["mode"] = np.select(
        [p["catu"] == "3", catv.isin([1, 80]), catv.isin([50, 60]),
         catv.isin([2, 30, 31, 32, 33, 34, 41, 42, 43]), catv.isin([7, 10]),
         catv.isin([13, 14, 15, 16, 17, 37, 38])],
        ["Pedestrian", "Bicycle", "E-scooter / PMD", "Powered two-wheeler", "Car / van", "Heavy vehicle / bus"],
        default="Other")

    # Keep only what the format checks need (not the full raw tables, to save memory)
    meta = {"columns": {t: list(d[t].columns) for t in TABLES},
            "codes": {(t, c): set(d[t][c].dropna()) for t, cols in CODE_COLS.items() for c in cols},
            "dep_metro": sorted(p["dep"].unique())}
    return p, meta

people, meta = {}, {}
for year in YEARS:
    people[year], meta[year] = build_year(year)

# --- Indicators: year x indicator (shares are % of metropolitan deaths of that year) ---
ONISR_KILLED_METRO = {2019: 3244, 2020: 2541, 2021: 2944, 2022: 3267, 2023: 3167, 2024: 3193}  # final ONISR figures

rows = {}
for year, p in people.items():
    k = p[p["killed"]]
    ped = k[k["mode"] == "Pedestrian"]
    rows[year] = {
        "killed_metro": len(k),
        "onisr_official": ONISR_KILLED_METRO[year],
        "departmental_road_%": (k["catr"] == "3").mean() * 100,
        "pedestrian_%": (k["mode"] == "Pedestrian").mean() * 100,
        "bicycle_%": (k["mode"] == "Bicycle").mean() * 100,
        "powered_two_wheeler_%": (k["mode"] == "Powered two-wheeler").mean() * 100,
        "aged_65+_%": (k["age"] >= 65).mean() * 100,                 # unknown age counts as "not 65+"
        "65+_among_pedestrians_killed_%": (ped["age"] >= 65).mean() * 100,
    }
trend = pd.DataFrame(rows).T.round(1)
trend["matches_onisr"] = trend["killed_metro"] == trend["onisr_official"]
trend

,killed_metro,onisr_official,departmental_road_%,pedestrian_%,bicycle_%,powered_two_wheeler_%,aged_65+_%,65+_among_pedestrians_killed_%,matches_onisr
2019,3244.0,3244.0,63.0,14.9,5.8,23.1,26.5,52.3,True
2020,2541.0,2541.0,61.7,15.3,7.0,22.8,25.7,52.4,True
2021,2944.0,2944.0,59.4,14.0,7.7,22.7,26.7,53.8,True
2022,3267.0,3267.0,59.1,14.9,7.5,22.0,27.5,45.5,True
2023,3167.0,3167.0,61.1,13.7,7.0,22.3,28.4,52.0,True
2024,3193.0,3193.0,62.3,14.2,7.0,22.5,28.7,49.6,True


In [7]:
# --- Format consistency across years: columns and codes that differ from one year to another ---
for t in TABLES:
    all_cols = set().union(*(meta[y]["columns"][t] for y in YEARS))
    for col in sorted(all_cols):
        missing = [y for y in YEARS if col not in meta[y]["columns"][t]]
        if missing:
            print(f"[column] {t}.{col} absent in {missing}")

for (t, c) in meta[YEARS[0]]["codes"]:
    sets = {y: meta[y]["codes"][(t, c)] for y in YEARS}
    common = set.intersection(*sets.values())
    for y, s in sets.items():
        if s != common:
            print(f"[code] {t}.{c} {y}: extra codes {sorted(s - common)}")

deps = {y: meta[y]["dep_metro"] for y in YEARS}
print("Metropolitan dep codes per year:", {y: len(v) for y, v in deps.items()})
for y in YEARS:
    if deps[y] != deps[YEARS[0]]:
        print(f"[dep] {y} differs from {YEARS[0]}: {sorted(set(deps[y]) ^ set(deps[YEARS[0]]))}")

[column] usagers.id_usager absent in [2019, 2020]
[code] caract.lum 2020: extra codes ['-1']
[code] caract.lum 2022: extra codes ['-1']
[code] caract.lum 2023: extra codes ['-1']
[code] caract.int 2020: extra codes ['-1']
[code] caract.int 2022: extra codes ['-1']
[code] caract.int 2023: extra codes ['-1']
[code] usagers.grav 2021: extra codes ['-1']
[code] usagers.grav 2022: extra codes ['-1']
[code] usagers.grav 2023: extra codes ['-1']
[code] usagers.sexe 2021: extra codes ['-1']
[code] usagers.sexe 2022: extra codes ['-1']
[code] usagers.sexe 2023: extra codes ['-1']
[code] usagers.sexe 2024: extra codes ['-1']
[code] vehicules.catv 2020: extra codes ['-1']
[code] vehicules.catv 2021: extra codes ['-1']
[code] vehicules.catv 2022: extra codes ['-1']
[code] vehicules.catv 2023: extra codes ['-1']
[code] vehicules.catv 2024: extra codes ['-1']
Metropolitan dep codes per year: {2019: 96, 2020: 96, 2021: 96, 2022: 96, 2023: 96, 2024: 96}
[dep] 2020 differs from 2019: ['01', '02', '03',

### Multi-year check (2019–2024)

- **Validation**: metropolitan deaths match the official ONISR figures **exactly for every year**.
- **Stable insights**: departmental roads 59–63% of deaths, powered two-wheelers 22–23%, pedestrians 14–15%,
  and about half of pedestrians killed are aged 65+ every year → the 2023 findings are not a one-year artefact.
- **Trends**: the share of people aged 65+ among deaths rises from 26.5% (2019) to 28.7% (2024);
  the cyclists' share increases after 2020 (5.8% → ~7%).
- **Format differences to handle in the pipeline**: `id_usager` absent in 2019–2020; département codes
  not zero-padded the same way across years ("1" vs "01"); `-1` codes appear in some years in `lum`, `int`,
  `grav`, `sexe`, `catv`.